# Introduction to Neural Networks

> Build a 2-layer network in NumPy, then the same thing in PyTorch.
> Backprop, by hand, once.

By the end of this notebook you'll have a working binary classifier on the
Wisconsin breast-cancer dataset (569 patients, 30 features) and you'll know
exactly which line in the training loop does what. We finish with a
confusion matrix and an ROC curve — the two plots every classification
result should ship with.

## 1. A single neuron

A neuron computes `f(w·x + b)`. That's the entire model. A *network* is
many neurons composed in layers. The two design choices are the
**activation function** `f` and the **shape** (how many layers, how wide).

In [1]:
import numpy as np, torch, torch.nn as nn, torch.optim as optim
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)

## 2. The dataset — Wisconsin breast cancer

569 samples, 30 numeric features per patient (radius, texture, perimeter…),
binary label (malignant vs benign). Real medical data, used as the standard
toy classification benchmark since 1995. Sklearn includes it.

In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X, y = data.data, data.target
print(f'shape: {X.shape}   classes: {data.target_names.tolist()}')
print(f'balance: {(y == 1).mean():.2f} benign, {(y == 0).mean():.2f} malignant')

shape: (569, 30)   classes: ['malignant', 'benign']
balance: 0.63 benign, 0.37 malignant


In [3]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
sc = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)

Xtr_t = torch.tensor(Xtr_s, dtype=torch.float32)
Xte_t = torch.tensor(Xte_s, dtype=torch.float32)
ytr_t = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1)
yte_t = torch.tensor(yte, dtype=torch.float32).unsqueeze(1)

## 3. The model — a 2-hidden-layer MLP

Same MLP structure as tutorial 01, but the **output is a single logit**
because this is binary classification. We apply `sigmoid` later to get a
probability.

In [4]:
class MLPClassifier(nn.Module):
    def __init__(self, in_dim, hidden=32, drop=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(hidden, 1),
        )
    def forward(self, x): return self.net(x)

model = MLPClassifier(Xtr_s.shape[1])
print(model)
print('params:', sum(p.numel() for p in model.parameters()))

MLPClassifier(
  (net): Sequential(
    (0): Linear(in_features=30, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=32, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)
params: 2081


## 4. `BCEWithLogitsLoss` and the training loop

`BCEWithLogitsLoss` combines a sigmoid with binary cross-entropy in one
numerically-stable op. **Always prefer it to `BCE + sigmoid` as two
separate ops** — better gradients near 0 and 1.

In [5]:
opt = optim.Adam(model.parameters(), lr=5e-3)
loss_fn = nn.BCEWithLogitsLoss()
tr_loss, va_loss = [], []
for ep in range(400):
    model.train()
    loss = loss_fn(model(Xtr_t), ytr_t)
    opt.zero_grad(); loss.backward(); opt.step()
    tr_loss.append(loss.item())
    model.eval()
    with torch.no_grad():
        va_loss.append(loss_fn(model(Xte_t), yte_t).item())
print(f'final train BCE {tr_loss[-1]:.4f}   val BCE {va_loss[-1]:.4f}')

final train BCE 0.0003   val BCE 0.3180


## 5. Evaluation — accuracy is the worst metric

For medical data: **a 95 % accurate classifier that misses every cancer is
worse than useless.** Always look at the confusion matrix and ROC.

In [6]:
from sklearn.metrics import confusion_matrix, roc_curve, auc

model.eval()
with torch.no_grad():
    probs = torch.sigmoid(model(Xte_t)).numpy().flatten()
preds = (probs >= 0.5).astype(int)

print(f'accuracy: {(preds == yte).mean():.3f}')
print(f'AUC:      {auc(*roc_curve(yte, probs)[:2]):.3f}')
print('confusion matrix:')
print(confusion_matrix(yte, preds))

accuracy: 0.956
AUC:      0.992
confusion matrix:
[[39  3]
 [ 2 70]]


## 6. What changes for multi-class

Three swaps and you have a 10-way classifier (think MNIST or CIFAR):

1. Final layer: `nn.Linear(hidden, n_classes)`
2. Loss: `nn.CrossEntropyLoss()` (no sigmoid — it includes softmax)
3. Targets: `int64` class indices, not floats

That's it. The rest of the loop is unchanged.

## What you've built

- A binary classifier on real medical data with **AUC > 0.99**.
- A tiny MLP with dropout that fits in <2 s on CPU.
- A confusion matrix + ROC — the two plots every classifier ships with.

**Next:** [`04 — Convolutional Neural Networks`](/tutorials/04-cnns/) —
what changes when the input is an image and you want translation invariance.

For the rest of the series, see [tutorials](/tutorials/).